In [288]:
# Imports
import numpy as np
import random

In [289]:
# Set-up

# Parameters
genome_size = 4
max_group_size = 6
starting_group_size = 3
group_number = 4
mutation_rate = 0.01
group_split_rate = 0.5
K = 3
K2 = 3

# Create initial population
initial_population = np.random.randint(0, 2, size = (group_number, max_group_size, genome_size), dtype = np.int8)
population = initial_population.copy()
display(population)

# Keep track of group sizes
group_sizes = np.full((group_number), starting_group_size, dtype = np.int8)
display(group_sizes)

# Keep track of group averages
# Mask 'True' only for indices less than the actual group size
mask = np.arange(max_group_size) < group_sizes[:, np.newaxis]
group_sums = np.sum(population * mask[:, :, np.newaxis], axis=1)
group_averages = group_sums / group_sizes[:, np.newaxis]

print(group_averages)

array([[[1, 0, 1, 1],
        [1, 1, 0, 0],
        [1, 1, 0, 0],
        [0, 0, 1, 0],
        [1, 0, 1, 0],
        [1, 1, 0, 0]],

       [[1, 1, 1, 1],
        [0, 1, 0, 0],
        [0, 1, 0, 0],
        [1, 0, 1, 1],
        [1, 1, 1, 0],
        [0, 1, 0, 0]],

       [[0, 1, 1, 1],
        [0, 0, 1, 0],
        [1, 1, 1, 0],
        [1, 1, 0, 0],
        [0, 0, 1, 1],
        [0, 0, 1, 1]],

       [[1, 0, 1, 0],
        [0, 1, 0, 1],
        [0, 0, 1, 1],
        [0, 0, 0, 0],
        [0, 1, 1, 0],
        [1, 1, 0, 0]]], dtype=int8)

array([3, 3, 3, 3], dtype=int8)

[[1.         0.66666667 0.33333333 0.33333333]
 [0.33333333 1.         0.33333333 0.33333333]
 [0.33333333 0.66666667 1.         0.33333333]
 [0.33333333 0.33333333 0.66666667 0.66666667]]


In [290]:
# Epistasis matrix

# Make empty matrix
epistasis_matrix = np.zeros((genome_size, K + K2), dtype = int)

for i in range(genome_size):
    # Sample K intragenomic partners (excluding focal locus i)
    available_loci = np.delete(np.arange(genome_size), i)  # All loci except i
    intragenomic_partners = np.random.choice(available_loci, size = K, replace = False)

    # Sample K2 intergenomic partners (can include focal locus i)
    intergenomic_partners = np.random.choice(genome_size, size = K2, replace = False)

    # Store in epistasis matrix
    epistasis_matrix[i, :K] = intragenomic_partners
    epistasis_matrix[i, K:] = intergenomic_partners


In [291]:
# Fitness matrix

beta_a = 0.5
beta_b = 0.5

fitness_matrix = np.random.beta(beta_a, beta_b, size=(genome_size, 2**(K+K2+1)))


In [292]:
# Advanced fitness calculation functions

def mobius_transform(fitness_values):
    num_coeffs = len(fitness_values)
    coefficients = np.zeros(num_coeffs, dtype=float)

    # Base case: coefficient for all-zeros corner (constant term)
    coefficients[0] = fitness_values[0]

    # Calculate remaining coefficients using inclusion-exclusion
    # Process in increasing order so dependencies are already computed
    for j in range(1, num_coeffs):
        subset_sum = 0.0

        # Sum coefficients for all proper subsets of j
        # A subset means: all bits set in l are also set in j
        for l in range(j):  # Only check l < j (proper subsets)
            # Bitwise AND: if l & j == l, then l is a subset of j
            if l == (l & j):
                subset_sum += coefficients[l]

        # Möbius inversion formula
        coefficients[j] = fitness_values[j] - subset_sum

    return coefficients

def calculateCoefficients(fitness_matrix, K, K2):

    N = fitness_matrix.shape[0]
    num_IESI = 2 ** (K + 1)
    num_multilinear_coeffs = 2 ** K2

    # Initialize coefficient tensor
    coefficients = np.zeros((N, num_IESI, num_multilinear_coeffs), dtype=float)

    # Process each locus
    for locus in range(N):
        # Process each IESI (Intragenomic Epistatic State Index)
        for iesi in range(num_IESI):
            # Extract the 2^K2 fitness values for this sub-hypercube
            # These are the fitness values for all possible states of the
            # K' group-average partners, given this IESI

            # INDEXING SCHEME:
            # fitness_matrix column index is a binary number with:
            # - High-order bits (positions K' to K+K'): IESI (focal + K partners)
            # - Low-order bits (positions 0 to K'-1): group corner state

            sub_hypercube = np.zeros(num_multilinear_coeffs, dtype=float)
            for group_corner in range(num_multilinear_coeffs):
                # Combine IESI (high bits) with group_corner (low bits)
                # Left shift IESI by K' positions, then OR with group_corner
                fitness_index = (iesi << K2) | group_corner
                sub_hypercube[group_corner] = fitness_matrix[locus, fitness_index]

            # Apply Möbius transform to get multilinear coefficients
            coefficients[locus, iesi, :] = mobius_transform(sub_hypercube)

    return coefficients

def kahan_sum(values):
    total = 0.0
    compensation = 0.0  # Tracks accumulated rounding error

    for value in values:
        # Compensate for previous rounding error
        y = value - compensation

        # Add to running total
        t = total + y

        # Calculate new rounding error
        # (t - total) is the rounded result of adding y
        # Subtracting y gives the rounding error
        compensation = (t - total) - y

        # Update total
        total = t

    return total

def evaluate_multilinear(coefficients, group_avg_values):
    num_coeffs = len(coefficients)
    terms = []  # Collect all terms before summing

    # Calculate each term in the multilinear expansion
    for j in range(num_coeffs):
        # Start with the coefficient
        term = coefficients[j]

        # Multiply by group_avg_values[k] for each bit k that is set in j
        # Example: if j=5 (binary 101), multiply by values[0] and values[2]
        for k in range(len(group_avg_values)):
            # Check if bit k is set in j using bitwise operations
            if j & (1 << k):  # (1 << k) creates a mask with bit k set
                term *= group_avg_values[k]

        terms.append(term)

    # Sum all terms using Kahan summation for numerical stability
    return kahan_sum(terms)

def calculateLocusFitness(locus, genome, group_avg_genome, epistasis_matrix, coefficients, K, K2):
    # Look up epistatic partners for this locus from epistasis matrix
    intragenomic_partners = epistasis_matrix[locus, :K]      # First K columns
    intergenomic_partners = epistasis_matrix[locus, K:]      # Last K' columns

    # Calculate IESI: binary index from focal locus + K intragenomic partners
    # Start with the focal locus bit (0 or 1)
    iesi = genome[locus]

    # Add bits from the K intragenomic partners
    # Partner i goes in bit position (i+1)
    for i, partner in enumerate(intragenomic_partners):
        iesi |= (genome[partner] << (i + 1))  # OR operation to set bits

    # Get group-average values at the K2 intergenomic partner loci
    # These are floats in [0,1], not binary!
    group_avg_values = group_avg_genome[intergenomic_partners]

    # Get the appropriate coefficients for this locus and IESI
    # This selects the right K'-dimensional sub-hypercube
    locus_coeffs = coefficients[locus, iesi, :]

    # Evaluate the multilinear expansion
    # This computes the interpolated fitness for this locus
    fitness_contribution = evaluate_multilinear(locus_coeffs, group_avg_values)

    return fitness_contribution

def calculateFitness(genome, group_avg_genome, epistasis_matrix, coefficients, K, K2):
    total_fitness = 0.0

    # Sum fitness contributions from all loci
    for locus in range(genome_size):
        fitness_contribution = calculateLocusFitness(
            locus, genome, group_avg_genome, epistasis_matrix, coefficients, K, K2
        )
        total_fitness += fitness_contribution

    # Return mean fitness (average across loci)
    return total_fitness / genome_size

In [293]:
# Test / init

coefficients = calculateCoefficients(fitness_matrix, K, K2)


In [294]:
# Fitness table

# Take population array and calculate the fitness for each indiviual in the initial population

# Calculate fitness of all genomes
# fitness_table = np.array([[calculateFitness(genome) for genome in group]for group in population])
fitness_table = np.array([
    [calculateFitness(genome, group_averages[i], epistasis_matrix, coefficients, K, K2) 
    for genome in group]
    for i, group in enumerate(population)
])

# Set 'empty' slots to 0
for i, size in enumerate(group_sizes):
    fitness_table[i, size:] = 0

fitness_array = fitness_table.flatten()
display(fitness_array)


# If a group splits or a individual is born, recalculate the fitnesses in that/those group(s)
# Than update this table 
# To not calculate the fitnesses of each individuals each timestep but only SOME individuals SOMEtimes

array([0.55977812, 0.49288437, 0.49288437, 0.        , 0.        ,
       0.        , 0.40749306, 0.44664898, 0.44664898, 0.        ,
       0.        , 0.        , 0.58820877, 0.50802552, 0.54893286,
       0.        , 0.        , 0.        , 0.50605642, 0.51154594,
       0.59263731, 0.        , 0.        , 0.        ])

In [295]:
# Helpfull functions

# Recalculate group average
def recalculateGroupAverage(group_id_input):
    # Mask 'True' only for indices less than the actual group size
    mask = (np.arange(max_group_size) < group_sizes[group_id_input])[:, np.newaxis]
    group_sums = np.sum(population[group_id_input] * mask, axis=0)
    group_averages[group_id_input] = group_sums / group_sizes[group_id_input]


# Recalculate fitness in one group
def recalculateGroupFitness(pop_input, group_id_input):
    updated_group_fitness = []
    for member_idx in range(group_sizes[group_id_input]):
        genome = pop_input[group_id_input, member_idx]
        # Use the advanced fitness function you defined earlier
        fit = calculateFitness(
            genome, 
            group_averages[group_id_input], 
            epistasis_matrix, 
            coefficients, 
            K, 
            K2
        )
        updated_group_fitness.append(fit)

    # 5. Map these values back into the global fitness_array
    start_idx = group_id_input * max_group_size
    end_idx = start_idx + max_group_size    
    
    # Create a fresh array for the group slice (initialized to 0)
    group_slice = np.zeros(max_group_size)
    group_slice[:group_sizes[group_id_input]] = updated_group_fitness
    
    # Update the global array
    fitness_array[start_idx:end_idx] = group_slice

In [296]:
def replaceMemberEvent(pop_input, group_id_input, newborn_genome_input):
    
    # Replace one unlucky member with newborn
    unlucky_index = np.random.choice(max_group_size)
    pop_input[group_id_input, [unlucky_index]] = newborn_genome_input
    print(f"Individual {unlucky_index} in group {group_id_input} died")
    
    # Recalculate group average genome
    recalculateGroupAverage(group_id_input)

    # Recalculate fitness
    recalculateGroupFitness(pop_input, group_id_input)

    return pop_input

def groupSplitEvent(pop_input, group_id_input, newborn_genome_input):
    # All groups except the current one
    left_over_groups = [i for i in range(group_number) if i != group_id_input]
    unlucky_group_index = np.random.choice(left_over_groups)
    # Set group size of unlucky group to 0
    group_sizes[unlucky_group_index] = 0

    # Boolean mask to split the group in a random way
    move_mask = np.random.choice([True, False], size = max_group_size, p = [0.5, 0.5])
        
    if not np.any(move_mask): # If all false, select 1 random to be true
        move_mask[np.random.randint(0, max_group_size)] = True
    elif np.all(move_mask):   # Vice versa
        move_mask[np.random.randint(0, max_group_size)] = False
            
    # Move those marked 'True' to new group
    moving_members = pop_input[group_id_input, move_mask, :]
    num_moving = len(moving_members)
    # Place them at the top of the new group
    pop_input[unlucky_group_index, 0:num_moving, :] = moving_members
    # Set group size to new amount
    group_sizes[unlucky_group_index] = num_moving
        
    # Move those marked 'False' to top of old group
    staying_members = pop_input[group_id_input, ~move_mask, :]
    num_staying = len(staying_members)
    # Place on top
    pop_input[group_id_input, 0:num_staying, :] = staying_members
    # Set group size to new amount
    group_sizes[group_id_input] = num_staying

    # Debug
    print(f"Group {group_id_input} split and replaced group {unlucky_group_index}.")
    print(f"{num_moving} individuals moved. {num_staying} stayed.")
    
    # Add newborn to old group
    # Place gemone copy below the taken spaces, overwriting the 'garbage' data
    pop_input[group_id_input, group_sizes[group_id_input]] = newborn_genome_input
    print(f"Reproduced member in group {group_id_input} at slot {group_sizes[group_id_input]}")

    # Increase group size so new individual is considered as 'actual' data
    group_sizes[group_id_input] += 1  
   
    # Recalculate group average genome of both groups
    recalculateGroupAverage(group_id_input)
    recalculateGroupAverage(unlucky_group_index)

    # Recalculate fitness in both groups
    recalculateGroupFitness(pop_input, group_id_input)  
    recalculateGroupFitness(pop_input, unlucky_group_index)

    return pop_input

def birthEvent(pop_input, group_id_input, newborn_genome_input):

    # Place gemone copy below the taken spaces, overwriting the 'garbage' data
    pop_input[group_id_input, group_sizes[group_id_input]] = newborn_genome_input
    print(f"Reproduced member in group {group_id_input} at slot {group_sizes[group_id_input]}")

    # Increase group size so new individual is considered as 'actual' data
    group_sizes[group_id_input] += 1
    new_size = group_sizes[group_id_input]
    
    # Recalculate group average genome
    recalculateGroupAverage(group_id_input)

    # Recalculate fitness
    recalculateGroupFitness(pop_input, group_id_input)

    return pop_input

def mutationEvent(newborn_genome_input):
    print('Genome before mutation event')
    print(newborn_genome_input)
    # Select random index in genome
    mutation_index = np.random.choice(genome_size)


    mutation_mask = np.random.choice([True, False], size = genome_size, p = [mutation_rate, 1-mutation_rate])

    # Flip those marked true
    newborn_genome_input[mutation_mask] = 1 - newborn_genome_input[mutation_mask]
    
    if not np.any(mutation_mask): # If all false, report no mutation
        print('No mutation!')
    else:
        print('Mutation occured!')

    print('Genome after mutation event')
    print(newborn_genome_input)
    return newborn_genome_input

In [297]:
# Reproduction function

def reproductionEvent(pop_input):
    # Select a random individual from fitness array based on weight
    probabilities = fitness_array / np.sum(fitness_array)
    selected_index = np.random.choice(len(fitness_array), p=probabilities)
    # Get group and member id
    group_id = selected_index//max_group_size
    member_id = selected_index%max_group_size

    print(f'Selected member {member_id} of group {group_id}')

    # Get genome of selected individual
    selected_genome = pop_input[group_id, member_id]
    genome_copy = selected_genome.copy()

    # Mutation event (has a chance within the function)
    mutationEvent(genome_copy)
    
    # Check if group is full
    if group_sizes[group_id] == max_group_size:
        # Either split or replace
        if random.random() > group_split_rate:
            replaceMemberEvent(pop_input, group_id, genome_copy)
        else:
            groupSplitEvent(pop_input, group_id, genome_copy)
    # Else just add new member to group
    else:
        birthEvent(pop_input, group_id, genome_copy)

In [298]:
display(group_averages)
print(fitness_array)
reproductionEvent(population)
display(population)
display(group_averages)
print(fitness_array)

array([[1.        , 0.66666667, 0.33333333, 0.33333333],
       [0.33333333, 1.        , 0.33333333, 0.33333333],
       [0.33333333, 0.66666667, 1.        , 0.33333333],
       [0.33333333, 0.33333333, 0.66666667, 0.66666667]])

[0.55977812 0.49288437 0.49288437 0.         0.         0.
 0.40749306 0.44664898 0.44664898 0.         0.         0.
 0.58820877 0.50802552 0.54893286 0.         0.         0.
 0.50605642 0.51154594 0.59263731 0.         0.         0.        ]
Selected member 2 of group 2
Genome before mutation event
[1 1 1 0]
No mutation!
Genome after mutation event
[1 1 1 0]
Reproduced member in group 2 at slot 3


array([[[1, 0, 1, 1],
        [1, 1, 0, 0],
        [1, 1, 0, 0],
        [0, 0, 1, 0],
        [1, 0, 1, 0],
        [1, 1, 0, 0]],

       [[1, 1, 1, 1],
        [0, 1, 0, 0],
        [0, 1, 0, 0],
        [1, 0, 1, 1],
        [1, 1, 1, 0],
        [0, 1, 0, 0]],

       [[0, 1, 1, 1],
        [0, 0, 1, 0],
        [1, 1, 1, 0],
        [1, 1, 1, 0],
        [0, 0, 1, 1],
        [0, 0, 1, 1]],

       [[1, 0, 1, 0],
        [0, 1, 0, 1],
        [0, 0, 1, 1],
        [0, 0, 0, 0],
        [0, 1, 1, 0],
        [1, 1, 0, 0]]], dtype=int8)

array([[1.        , 0.66666667, 0.33333333, 0.33333333],
       [0.33333333, 1.        , 0.33333333, 0.33333333],
       [0.5       , 0.75      , 1.        , 0.25      ],
       [0.33333333, 0.33333333, 0.66666667, 0.66666667]])

[0.55977812 0.49288437 0.49288437 0.         0.         0.
 0.40749306 0.44664898 0.44664898 0.         0.         0.
 0.62160354 0.41537046 0.54557785 0.54557785 0.         0.
 0.50605642 0.51154594 0.59263731 0.         0.         0.        ]
